### Bronze: GTFS-realtime (VehiclePositions + TripUpdates)
##### Structurally different from static Bronze: GTFS-realtime is protobuf served live over HTTP, not a file for Auto Loader to discover, so this notebook polls it directly.
##### Runs a fixed-duration poll loop and finishes (same `availableNow` reasoning as static Bronze) -- scheduled to re-run every few minutes rather than run continuously.
##### Same multi-mode design: both modes write into the same tables via `gtfs_mode`. Buses' endpoints are wired up but only `sydneytrains` is validated tonight.

In [0]:
%run ../../commonsetup/Setup-Common

In [0]:
import time
from datetime import datetime, timezone

import requests
from google.transit import gtfs_realtime_pb2
from pyspark.sql import functions as F



### 1: Config
##### CONFIRM both templates against your registered endpoints -- VehiclePositions and TripUpdates are separate URLs per mode, not one combined feed.

In [0]:
GTFS_REALTIME_VEHICLE_POS_URL_TEMPLATE = "https://api.transport.nsw.gov.au/v2/gtfs/vehiclepos/{mode}"  # CONFIRM
GTFS_REALTIME_TRIP_UPDATES_URL_TEMPLATE = "https://api.transport.nsw.gov.au/v2/gtfs/realtime/{mode}"    # CONFIRM

REQUEST_TIMEOUT_SECONDS = 30
MAX_RETRIES_PER_POLL = 2
RETRY_BACKOFF_SECONDS = 5

# How long this task runs before finishing (bounded, not infinite) and how
# often it polls within that window. Schedule the Workflow task itself to
# re-run every POLL_DURATION_SECONDS (or a bit more) so coverage is
# continuous across runs.
POLL_DURATION_SECONDS = 5 * 60   # 5 minutes per task run. For test run lets make it 2 minutes
POLL_INTERVAL_SECONDS = 30       # poll every 30s within that window

api_key = dbutils.secrets.get(scope="tfnsw-secrets", key="tfnsw-api-key")

# Buses is wired up but not validated yet -- extend to GTFS_MODES once it is.
ACTIVE_MODES_TONIGHT = ["sydneytrains"]



### 2: Fetch + parse one mode's feed, isolated per (mode, feed_type)
##### A failed poll is just a skipped interval, not fatal -- GTFS-realtime is inherently best-effort (a position 30s later supersedes a missed one), unlike static Bronze where a missing file is a real gap.

In [0]:
def fetch_feed(url: str, api_key: str) -> gtfs_realtime_pb2.FeedMessage:
    headers = {"Authorization": f"apikey {api_key}"}
    last_error = None
    for attempt in range(1, MAX_RETRIES_PER_POLL + 1):
        try:
            response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
            response.raise_for_status()
            feed = gtfs_realtime_pb2.FeedMessage()
            feed.ParseFromString(response.content)
            return feed
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES_PER_POLL:
                time.sleep(RETRY_BACKOFF_SECONDS)
    raise RuntimeError(f"Poll failed after {MAX_RETRIES_PER_POLL} attempts: {last_error}")




In [0]:
from google.protobuf.json_format import MessageToDict
import json

def parse_vehicle_positions(feed: gtfs_realtime_pb2.FeedMessage, mode: str, poll_ts: datetime) -> list:
    rows = []
    for entity in feed.entity:
        if not entity.HasField("vehicle"):
            continue
        rows.append({
            "gtfs_mode": mode,
            "entity_id": entity.id,
            "poll_timestamp": poll_ts,
            "raw_json": json.dumps(MessageToDict(entity.vehicle, preserving_proto_field_name=True)),
        })
    return rows

In [0]:
def parse_trip_updates(feed: gtfs_realtime_pb2.FeedMessage, mode: str, poll_ts: datetime) -> list:
    rows = []
    for entity in feed.entity:
        if not entity.HasField("trip_update"):
            continue
        tu = entity.trip_update
        stop_time_updates = [
            {
                "stop_sequence": stu.stop_sequence if stu.HasField("stop_sequence") else None,
                "stop_id": stu.stop_id if stu.HasField("stop_id") else None,
                "arrival_delay": stu.arrival.delay if stu.HasField("arrival") and stu.arrival.HasField("delay") else None,
                "departure_delay": stu.departure.delay if stu.HasField("departure") and stu.departure.HasField("delay") else None,
            }
            for stu in tu.stop_time_update
        ]
        rows.append({
            "gtfs_mode": mode,
            "entity_id": entity.id,
            "trip_id": tu.trip.trip_id if tu.HasField("trip") else None,
            "route_id": tu.trip.route_id if tu.HasField("trip") else None,
            "start_date": tu.trip.start_date if tu.HasField("trip") else None,
            "vehicle_id": tu.vehicle.id if tu.HasField("vehicle") else None,
            "stop_time_updates": stop_time_updates,
            "poll_timestamp": poll_ts,
        })
    return rows



### 3: Bounded poll loop
##### Collects in memory across the window, writes once at the end -- avoids a Delta small-file write every 30s.

In [0]:
collected = {
    ("vehicle_positions", mode): [] for mode in ACTIVE_MODES_TONIGHT
}
collected.update({
    ("trip_updates", mode): [] for mode in ACTIVE_MODES_TONIGHT
})

poll_errors = {}  # (feed_type, mode) -> count of failed polls

window_end = time.monotonic() + POLL_DURATION_SECONDS

while time.monotonic() < window_end:
    poll_ts = datetime.now(timezone.utc)

    for mode in ACTIVE_MODES_TONIGHT:
        # Vehicle positions
        try:
            vp_feed = fetch_feed(GTFS_REALTIME_VEHICLE_POS_URL_TEMPLATE.format(mode=mode), api_key)
            collected[("vehicle_positions", mode)].extend(parse_vehicle_positions(vp_feed, mode, poll_ts))
        except Exception as e:
            poll_errors[("vehicle_positions", mode)] = poll_errors.get(("vehicle_positions", mode), 0) + 1
            print(f"[{mode}] vehicle_positions poll failed: {e}")

        # Trip updates
        try:
            tu_feed = fetch_feed(GTFS_REALTIME_TRIP_UPDATES_URL_TEMPLATE.format(mode=mode), api_key)
            collected[("trip_updates", mode)].extend(parse_trip_updates(tu_feed, mode, poll_ts))
        except Exception as e:
            poll_errors[("trip_updates", mode)] = poll_errors.get(("trip_updates", mode), 0) + 1
            print(f"[{mode}] trip_updates poll failed: {e}")

    time.sleep(POLL_INTERVAL_SECONDS)

print("Poll window complete. Row counts collected:")
for key, rows in collected.items():
    print(f"  {key}: {len(rows)} rows")



### 4: Write to Bronze -- append-only, same principle as static
##### `stop_time_updates` stays nested (array<struct>) here -- flattening to one row per stop is Silver's job, not raw capture's.

In [0]:
def rescue_and_align_schema(df, target_table: str):
    """
    Aligns df to the target Delta table's existing schema instead of letting
    it merge/evolve silently. Any column not already in the target table, or
    whose type doesn't match, is moved into a `_rescued_data` JSON column and
    dropped from the main write -- nothing is lost, nothing is silently
    merged in, and the table's schema never drifts without a deliberate
    change to this notebook.
    """
    if "_rescued_data" not in df.columns:
        df = df.withColumn("_rescued_data", F.lit(None).cast("string"))
 
    if not spark.catalog.tableExists(target_table):
        # First-ever write for this table: whatever shape df has now becomes
        # the baseline schema (including the empty _rescued_data column, so
        # every future run has somewhere to put surprises).
        return df
 
    target_fields = {
        f.name: f.dataType for f in spark.table(target_table).schema
        if f.name != "_rescued_data"
    }
 
    stable_cols, rescue_cols = [], []
    for field in df.schema:
        if field.name == "_rescued_data":
            continue
        if field.name in target_fields and field.dataType == target_fields[field.name]:
            stable_cols.append(field.name)
        else:
            rescue_cols.append(field.name)
 
    if not rescue_cols:
        return df.select(*stable_cols, "_rescued_data")
 
    print(f"  -> rescuing columns not in {target_table}'s current schema: {rescue_cols}")
    rescued_expr = F.to_json(F.struct(*[F.col(c) for c in rescue_cols]))
    return df.select(*[F.col(c) for c in stable_cols], rescued_expr.alias("_rescued_data"))

In [0]:
run_results = {}  # (feed_type, mode) -> "success" | "failed" | "no_data"

for (feed_type, mode), rows in collected.items():
    target_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_gtfs_{feed_type}"

    if not rows:
        print(f"({feed_type}, {mode}): no rows collected this run")
        run_results[(feed_type, mode)] = "no_data"
        continue

    try:
        df = (
            spark.createDataFrame(rows)
            .withColumn("ingestion_timestamp", F.current_timestamp())
        )
        df = rescue_and_align_schema(df, target_table)
        df.write.format("delta").mode("append").saveAsTable(target_table)
        print(f"({feed_type}, {mode}): wrote {len(rows)} rows -> {target_table}")
        run_results[(feed_type, mode)] = "success"
    except Exception as e:
        print(f"({feed_type}, {mode}): write failed: {e}")
        run_results[(feed_type, mode)] = "failed"

### 5: Readiness signal
##### "no_data" for a whole poll window is treated as seriously as a write failure -- an active feed returning nothing for 5 minutes usually means the endpoint/auth is wrong, not that no vehicles are running.

In [0]:
problem_pairs = {k for k, v in run_results.items() if v in ("failed", "no_data")}

run_log_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_streaming_run_log"
run_log_df = spark.createDataFrame(
    [(feed_type, mode, outcome) for (feed_type, mode), outcome in run_results.items()],
    schema="feed_type STRING, gtfs_mode STRING, outcome STRING",
).withColumn("run_timestamp", F.current_timestamp())
run_log_df.write.format("delta").mode("append").saveAsTable(run_log_table)

if problem_pairs:
    print(f"Problem (feed_type, mode) pairs this run: {sorted(problem_pairs)}")
    print(f"Poll error counts: {poll_errors}")
else:
    print("All active feed/mode pairs ingested successfully this run.")

In [0]:
%sql
select * from bronze_gtfs_vehicle_positions;